In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, make_scorer, fbeta_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV, TunedThresholdClassifierCV
from joblib import dump
import numpy as np

pd.set_option("display.float_format", lambda x: "%0.3f" % x)
np.set_printoptions(suppress=True)

def print_line(): print("-" * 15)

In [3]:
data = pd.read_csv("Data/processed_data")
data

,annual_income,loan_amount,int_rate,home_ownership,purpose,is_loss,DTI,annual_income_ru,loan_ammount_ru,int_rate_ru
0,30000.000,2500,0.153,RENT,car,1,0.083,444000.000,37000.000,0.219
1,48000.000,3000,0.186,RENT,car,0,0.062,710400.000,44400.000,0.267
2,50000.000,12000,0.160,RENT,car,1,0.240,740000.000,177600.000,0.229
3,42000.000,4500,0.106,MORTGAGE,car,0,0.107,621600.000,66600.000,0.153
4,83000.000,3500,0.060,MORTGAGE,car,0,0.042,1228400.000,51800.000,0.086
...,...,...,...,...,...,...,...,...,...,...
38568,100000.000,24250,0.130,MORTGAGE,other,0,0.242,1480000.000,358900.000,0.186
38569,50000.000,25200,0.135,RENT,other,0,0.504,740000.000,372960.000,0.193
38570,65000.000,25000,0.175,RENT,other,0,0.385,962000.000,370000.000,0.251
38571,368000.000,24000,0.182,RENT,other,0,0.065,5446400.000,355200.000,0.262


In [4]:
X = data[["annual_income_ru", "loan_ammount_ru", "int_rate_ru", "DTI","home_ownership", "purpose"]] 
Y = data["is_loss"]

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.15,stratify=Y, random_state=42)
print(f"X_train: {X_train.shape}\nX_test: {X_test.shape}\nY_train: {Y_train.shape}\nY_test: {Y_test.shape}")
print(X_train.head())

X_train: (32787, 6)
X_test: (5786, 6)
Y_train: (32787,)
Y_test: (5786,)
       annual_income_ru  loan_ammount_ru  int_rate_ru   DTI home_ownership  \
13836       1184000.000       229400.000        0.158 0.194       MORTGAGE   
22927        666000.000       296000.000        0.242 0.444       MORTGAGE   
25687        740000.000       222000.000        0.147 0.300       MORTGAGE   
3452         828800.000        71040.000        0.200 0.086           RENT   
3508         444000.000       148000.000        0.252 0.333           RENT   

                  purpose  
13836  Debt consolidation  
22927  Debt consolidation  
25687               other  
3452          credit card  
3508          credit card  


Преобразуем числовые данные в единый масштаб (Нормализация). Долго (Заменено готовой библиотекой)

In [5]:
# X_train_norm = X_train.copy()
# X_test_norm = X_test.copy()
# for col in X_train.columns[:3]:
#     X_train_norm[col] = X_train_norm[col].apply(lambda val: (val - min) / (max - min))[col]
#     X_test_norm[col] = X_test_norm[col].apply(lambda val: (val - min) / (max - min))[col]
# X_train_norm.to_csv("Data/X_train_norm", index= False)
# X_test_norm.to_csv("Data/X_test_norm", index= False)

In [6]:
# X_train_norm = pd.read_csv("Data/X_train_norm")
# X_test_norm = pd.read_csv("Data/X_test_norm")

In [ ]:
# model = RandomForestClassifier(n_estimators= 100, max_depth= 4,
#                                random_state=42,
#                                class_weight=dict(enumerate(class_weights)))
# numerical_transformer = Pipeline(steps=[
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', StandardScaler())
# ])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, [4,5]),
        ('num', 'passthrough', [0,1,2,3])
    ]
)

# class_weights = compute_class_weight(class_weight="balanced", classes = np.unique(Y_train), y=Y_train)

"""dict(enumerate(class_weights))"""

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ("classifier", TunedThresholdClassifierCV(estimator=RandomForestClassifier(n_estimators=500, 
                                          class_weight = "balanced_subsample",
                                          criterion="entropy",
                                          max_features=4,random_state=42),scoring="f1",
                                          cv=StratifiedKFold(n_splits=10,shuffle=True,random_state=42)))
])

In [14]:
f_score = make_scorer(fbeta_score, betta = 1.5)
param_dist = {
    "classifier__min_samples_split": range(5,80),
    "classifier__min_samples_leaf": range(5, 80),
    "classifier__max_depth":range(5, 12), 
}
random_search = RandomizedSearchCV(estimator=pipeline, 
                            param_distributions= param_dist,
                             n_iter= 100,
                             scoring=f_score,
                             cv=StratifiedKFold(random_state=42, shuffle=True),
                             random_state=42,
                             n_jobs=-1,
                             verbose=1)

random_search.fit(X_train, Y_train)
print(random_search.best_params_)
#Итог
# classifier__min_samples_split = 11
# classifier__min_samples_leaf = 17
# classifier__max_depth = 15

Fitting 5 folds for each of 100 candidates, totalling 500 fits


c:\Github\Credit-s-RIsk-Prediction\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
c:\Github\Credit-s-RIsk-Prediction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


{'classifier__min_samples_split': 50, 'classifier__min_samples_leaf': 65, 'classifier__max_depth': 7}


In [35]:
min_samples_split = 50
min_samples_leaf = 65
max_depth = 7
best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", TunedThresholdClassifierCV(estimator=RandomForestClassifier(n_estimators=500, 
                                          class_weight = "balanced_subsample",
                                          criterion="entropy",
                                          max_features=4,random_state=42,min_samples_split=min_samples_split,
                                          min_samples_leaf=min_samples_leaf,
                                          max_depth= max_depth),scoring="f1",
                                          cv=StratifiedKFold(n_splits=10,shuffle=True,random_state=42)))
])

In [36]:
metrics = ["precision", "recall", "roc_auc", "f1"]
cross_val_result = cross_validate(best_pipeline,
                                    X_train, Y_train,
                                   cv=StratifiedKFold(n_splits=10,shuffle=True, random_state=42),
                                   scoring=metrics,
                                   return_train_score=True,
                                   n_jobs=-1)

KeyboardInterrupt: 

Лишний Раз не запускать!

In [11]:
checking_metrics = ['test_roc_auc', 'train_roc_auc', "test_f1", "train_f1", "train_precision", "test_precision",
               "train_recall", "test_recall"]
def print_dict(results: dict):
    for key, value in results.items():
        print(key + ": " + f"{value}")

def get_crossval_results()->dict:
    result = {}
    for metric in checking_metrics:
        result[metric] = float(cross_val_result.get(metric).mean())
    return result
metrics_results = []

In [31]:
if len(metrics_results) == 0:
    metrics_results = [get_crossval_results()]
    print_dict(metrics_results[0])
elif len(metrics_results) == 1:
    metrics_results.append(get_crossval_results())
    for metric in checking_metrics:
        if metrics_results[1][metric] > metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬆️")
        elif metrics_results[1][metric] < metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬇️")
        else: 
            print(metric + ": " + f"{metrics_results[1][metric]}")
    
elif len(metrics_results) == 2:
    metrics_results[0] = metrics_results[1]
    metrics_results[1] = get_crossval_results()
    for metric in checking_metrics:
        if metrics_results[1][metric] > metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬆️")
        elif metrics_results[1][metric] < metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬇️")
        else: 
            print(metric + ": " + f"{metrics_results[1][metric]}")

test_roc_auc: 0.6797282103620572⬆️
train_roc_auc: 0.7060165525358613⬇️
test_f1: 0.31752225567280623⬆️
train_f1: 0.33180527482655287⬇️
train_precision: 0.21879886299474668⬇️
test_precision: 0.20943928066593767⬇️
train_recall: 0.6862758798052194⬇️
test_recall: 0.6563035465958709⬆️


In [37]:
best_pipeline.fit(X_train, Y_train)
prediction = best_pipeline.predict(X_test)
f1 = f1_score(Y_test, prediction)
recall = recall_score(Y_test, prediction)
precision = precision_score(Y_test, prediction)
print(f"f1: {f1}\nRecall: {recall}\nPrecision: {precision}")

f1: 0.33063733419605307
Recall: 0.63875
Precision: 0.22304670449585334


Подбираем лучшие параметры для модели (ОЧЕНЬ ДОЛГО). Результат: n_estimators = 106, max_depth = 4.Заменено библиотекой

In [ ]:
# best_f = 0
# for estimators in range(10, 150, 3):
#     for depth in range(3, 15):
#         test_model = RandomForestClassifier(n_estimators=estimators, max_depth= depth, 
#                                             random_state=42,
#                                             class_weight=dict(enumerate(class_weights)))
#         test_model.fit(X_train_norm, Y_train)
#         f_score = fbeta_score(Y_test, test_model.predict(X_test_norm), beta=1.5)
#         if f_score > best_f:
#             best_f = f_score
#             best_estimators = estimators
#             best_max_depth = depth

# print(f"Лучший параметр estimators: {best_estimators}\nЛучший параметр max_depth: {best_max_depth}")

In [31]:
# prediction = model.predict(X_test)
# prediction = pd.DataFrame(prediction, columns=['Prediction'])
# results = pd.concat([X_test, prediction["Prediction"], Y_test], axis= 1, join="inner")
# results.rename(columns={"is_loss": "Real Value"}, inplace= True) 
# results[(results["Real Value"] == 1) & (results["Prediction"] == 1)]

In [38]:
model_path = "models/RandomForest.joblib"
dump(best_pipeline, model_path)

['models/RandomForest.joblib']